# Exploring Lorebase
Run the cells one by one. Start the services first with `docker compose up -d redis qdrant`, then run `python -m app.ingest`.

- **Embedding**: text becomes a vector, and texts with similar meaning get nearby vectors.
- **Vector search (Qdrant)**: finds the stored vectors closest to the question's vector.
- **Cache (Redis)**: repeated questions are answered without searching again.

In [ ]:
from pathlib import Path
from app import config
from app.core import chunk_text

chunks = chunk_text(Path(config.DATA_PATH).read_text(encoding="utf-8"))
print(len(chunks), "chunks")
for i, c in enumerate(chunks):
    print(i, c[:80], "...")

## 1. Embeddings
Look at the vector shape and compare the similarity of three sentences.

In [ ]:
import numpy as np
from app.main import get_embedder

embedder = get_embedder()
a, b, c = [np.array(v) for v in embedder.embed([
    "I can build Android apps", "I am good with Flutter for mobile", "A tasty fried rice recipe"])]
cos = lambda x, y: float(x @ y / (np.linalg.norm(x) * np.linalg.norm(y)))
print("dimensions:", a.shape)
print("Android vs Flutter   :", round(cos(a, b), 3))
print("Android vs fried rice:", round(cos(a, c), 3))

## 2. Search in Qdrant
Change the question and watch the scores and passages change. Indonesian works too.

In [ ]:
from app.main import get_qdrant

client = get_qdrant()

def search(question, k=3):
    vector = next(iter(embedder.embed([question]))).tolist()
    for p in client.query_points(config.COLLECTION, query=vector, limit=k).points:
        print(round(p.score, 3), "|", p.payload["text"][:100])

search("What is a vector database?")
search("Apa fungsi Redis?")

## 3. Cache in Redis
Compare the time of a cold search and a cached one through the API.

In [ ]:
import requests

for attempt in (1, 2):
    r = requests.post("http://localhost:8000/ask", json={"question": "What is Docker Compose?"}).json()
    print(attempt, "cached:", r["cached"], "| ms:", r["elapsed_ms"])

## Experiments to try
1. Change `max_chars` in `chunk_text` to 150, run `python -m app.ingest` again, and compare the search results.
2. Add a paragraph to `data/knowledge.md`, re-index, and search for it.
3. Stop Redis (`docker compose stop redis`) and call `/ask`. The API keeps working, just without the cache.